<center>
    <img src="https://raw.githubusercontent.com/dvgodoy/PyTorch101_ODSC_Europe2020/master/images/linear_dogs.jpg" width="900">
</center>  

# Yet another math for ML course

# Домашнее задание 2: SVD и спектры изображений

**Социальный бонус [1 балл]:** у нас онлайн-магистратура, но это не отменяет движа. Сходите куда-нибудь с однокурсниками и прикрепите фото с этого похода сюда.

## Формат проверки работы: AI-агент + ассистент

Что это значит для вас:

* тетрадка должна выполняться сверху вниз без ошибок — перед сдачей сделайте `Restart & Run All`;
* не переименовывайте функции и переменные, которые заданы в заготовках;
* ответы на вопросы пишите в отведённые ячейки после «Ответ:», коротко и по существу. Ответы — часть задания и оцениваются наравне с кодом: работающий код с пустым «Ответ:» полного балла не получает;
* у графиков должны быть подписаны оси и заголовки — агент читает картинки, но не умеет угадывать, что на них;
* комментарии проверяющего придут вместе с оценкой. Если вы с ней не согласны — напишите ассистенту, указав конкретное задание и что именно, по вашему мнению, проверено неверно;
* последнее слово в выставлении оценки будет за ассистентом. 

> В какие-то из работ ассистенты придут сами для перепроверки баллов.

Пользоваться ChatGPT, Claude и прочими как справочником и напарником можно.

Сдавать сгенерированное целиком решение, которое вы не понимаете, — нельзя. Вы обязаны разобраться в каждой строчке своего кода: часть студентов будет вызвана на защиту домашки, и если защита провалится, оценка за работу обнулится.

-------------------

В этой домашке мы будем изучать сингулярное разложение матрицы на примере изображений.

Главная идея домашки:

> **Сингулярные числа — это отпечаток структуры данных.**

Мы возьмём четыре очень разных изображения: белый шум, QR-код, фотографию и страницу с текстом.

Для каждого изображения мы построим SVD, посмотрим на спектр сингулярных чисел, попробуем восстановить изображение по первым компонентам и поймём, какие изображения хорошо сжимаются низкоранговыми приближениями, а какие — плохо.

### После этой домашки хочется, чтобы вы могли ответить на вопросы:

* что такое SVD и почему его можно понимать как сумму простых картинок ранга 1;
* что такое сингулярные числа и почему их спад говорит о наличии структуры;
* почему фотография обычно хорошо сжимается, а белый шум — плохо;
* как выбирать число компонент;
* как связаны SVD, спектральное разложение, низкоранговые приближения и PCA.


# Часть 0: готовим картинки

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage import data, color, transform, img_as_float

np.random.seed(42)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.size"] = 12

def to_gray_square(img, size=256):
    img = img_as_float(img)
    if img.ndim == 3:
        img = color.rgb2gray(img)
    img = transform.resize(img, (size, size), anti_aliasing=True)
    return np.clip(img, 0, 1)

# Делаем QR-like картинку без внешних библиотек
def make_qr_like(size=256, blocks=29, seed=7):
    rng = np.random.default_rng(seed)
    grid = rng.integers(0, 2, size=(blocks, blocks)).astype(float)

    def finder(i, j, w=7):
        grid[i:i+w, j:j+w] = 1
        grid[i+1:i+w-1, j+1:j+w-1] = 0
        grid[i+2:i+w-2, j+2:j+w-2] = 1

    finder(1, 1)
    finder(1, blocks - 8)
    finder(blocks - 8, 1)

    img = np.kron(grid, np.ones((size // blocks + 1, size // blocks + 1)))
    img = img[:size, :size]
    return img

In [ ]:
size = 256
images = {
    "A: белый шум": np.random.rand(size, size),
    "B: QR-like код": make_qr_like(size=size),
    "C: фоточка": to_gray_square(data.chelsea(), size=size),
    "D: страница с текстом": to_gray_square(data.page(), size=size),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

Перед нами четыре картинки. Давайте включим свою интуицию и ответим на вопросы:

- Какое изображение легче всего сжать с помощью SVD, если оставить мало компонент?  
- Какое изображение сжать сложнее всего?

Под «сжать» мы будем понимать следующее: заменить исходную матрицу картинки на матрицу меньшего ранга.

> Важно: задание 0 не оценивается. В нём можно ошибиться. Пожалуйста, напишите свою настоящую интуицию и не исправляйте её в корректную после решения задания.

__Задание 0:__ Расположите изображения от самого легко сжимаемого к самому плохо сжимаемому.

**Ваш ответ:**

1. ...
2. ...
3. ...
4. ...

Коротко объясните, почему вы думаете именно так.

# Часть 1: разбираемся как устроено SVD

Введём *норму Фробениуса* матрицы как

$$||A||_{F} = \sqrt{\text{tr}(A^TA)} = \sqrt{\sum\limits_{i,j}a^2_{i,j}}$$

Иными словами, это обычное евклидово расстояние на пространстве, которое получается, если все матрицы вытянуть в длинные векторы.

Зададимся вопросом: как найти матрицу $A_{r}$ ранга $r$, наименее отличающуюся от $A$ по норме Фробениуса (то есть для которой норма разности $||A - A_{r}||_{F}$ минимальна). Оказывается, это можно сделать с помощью сингулярного разложения.


Пусть изображение — это матрица $A \in \mathbb{R}^{m \times n}$, где элемент $a_{ij}$ отвечает за яркость пикселя.

Сингулярное разложение говорит, что любую матрицу можно представить в виде

$$
A = U \Sigma V^T,
$$

где

* $U$ и $V$ — ортогональные матрицы;
* $\Sigma$ — диагональная матрица с неотрицательными числами;
* числа $\sigma_1 \ge \sigma_2 \ge \ldots \sigma_r \ge 0$ называются **сингулярными числами**.

Обозначим через $u_1,\ldots, u_k$ столбцы матрицы $U$, а через $v_1, \ldots, v_k$ столбцы матрицы $V$. Тогда имеет место равенство

$$
A = \sigma_1 u_1 v_1^T + \sigma_2 u_2 v_2^T + \ldots + \sigma_r u_r v_r^T.
$$

Каждое слагаемое $\sigma_i u_i v_i^T$ — это картинка ранга 1. Если оставить только первые $k$ слагаемых, получится приближение

$$
A_k = \sum_{i=1}^k \sigma_i u_i v_i^T.
$$

Теорема Эккарта — Янга говорит, что:

> Среди всех матриц ранга не больше $k$ матрица $A_k$ является лучшим приближением к $A$ в норме Фробениуса.


Есть несколько способов сделать в Питоне сингулярное разложение; мы предлагаем использовать

`U, s, Vt = np.linalg.svd(A, full_matrices=False)`

Для ознакомления с особенностями этой функции рекомендуем обратиться к [документации](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html) (в частности, обратите внимание на то, какие именно объекты она возвращает).

__(0.5 балла) Задание 1:__ Посчитайте SVD для всех изображений. Для этого допишите функцию `compute_svd` и примените её.

In [ ]:
def compute_svd(A):
    # YOUR CODE HERE
    # ʕ•ᴥ•ʔ
    pass

svds = {}
for name, img in images.items():
    # YOUR CODE HERE
    # [✖‿✖]
    pass

Сингулярные числа часто называют **спектром матрицы**. Если сингулярные числа быстро убывают, значит большая часть структуры изображения описывается небольшим числом компонент. Если сингулярные числа убывают медленно, значит информация размазана по многим направлениям.

__(1 балл) Задание 2:__ Для каждого изображения постройте два графика:

- обычный график сингулярных чисел;
- график в логарифмической шкале по оси $y$.

In [ ]:
def plot_singular_values(svds):
    # YOUR CODE HERE
    # ┌(ಠ_ಠ)┘
    pass

plot_singular_values(svds)

__Ответьте на вопросы!__

У какого изображения сингулярные числа падают быстрее всего?

__Ответ:__

У какого изображения спектр самый «плоский»?

__Ответ:__

Что это говорит о структуре изображения?

__Ответ:__

Совпало ли это с вашей интуицией из вопроса 0?

__Ответ:__

Теперь восстановим изображения по первым $k$ компонентам:

$$
A_k = U_k \Sigma_k V_k^T.
$$

Если $k$ маленькое, мы храним не всю матрицу $A$, а только:

* первые $k$ столбцов $U$;
* первые $k$ сингулярных чисел;
* первые $k$ строк $V^T$.

Для изображения размера $m \times n$ это требует $k \cdot (m + n + 1)$ чисел вместо $m \cdot n$.

__(1 балл) Задание 3:__  Реализуйте восстановление по первым k компонентам. Для каждого изображения нарисуйте приближения при

$$
k \in \{1, 5, 10, 20, 50, 100\}.
$$

In [ ]:
# Оценка, во сколько раз исходная матрица хранится дороже, чем низкоранговое приближение ранга k.
def compression_ratio(shape, k):
    m, n = shape
    original = m * n
    compressed = k * (m + n + 1)
    return original / compressed

def reconstruct_from_svd(U, s, Vt, k):
    # YOUR CODE HERE
    # ʕ•ᴥ•ʔ
    pass

def show_reconstructions(name, img, svd, ks=(1, 5, 10, 20, 50, 100)):
    U, s, Vt = svd

    fig, axes = plt.subplots(1, len(ks) + 1, figsize=(16, 3))

    axes[0].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("original")
    axes[0].axis("off")

    for ax, k in zip(axes[1:], ks):
        # YOUR CODE HERE: восстановите изображение по первым k компонентам
        # ( .-. )
        Ak = None

        ax.imshow(Ak, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"k={k}")
        ax.axis("off")

    fig.suptitle(name, y=1.05)
    plt.tight_layout()
    plt.show()

for name, img in images.items():
    show_reconstructions(name, img, svds[name])

__Ответьте на вопросы!__

Какие изображения становятся узнаваемыми уже при маленьких $k$?

__Ответ:__

Какие остаются плохо узнаваемыми даже при $k = 100$?

__Ответ:__

Почему белый шум почти невозможно хорошо сжать низким рангом?

__Ответ:__

Почему QR код может вести себя не так, как обычная фотография?

__Ответ:__

Теорема Эккарта — Янга говорит, что ошибка лучшего приближения ранга $k$ в норме Фробениуса равна

$$
\|A - A_k\|_F^2 = \sum_{i > k} \sigma_i^2.
$$

То есть по спектру можно заранее понять, насколько хорошо будет восстановление.

__(0.5 балла) Задание 4:__ посчитайте относительную ошибку восстановления.

Будем считать

$$
\operatorname{rel\_err}(k) = \frac{\|A - A_k\|_F}{\|A\|_F}.
$$

Её можно считать двумя способами:

1. напрямую через матрицы;
2. через сингулярные числа.

Реализуйте оба способа и проверьте, что они дают одинаковый результат.

In [ ]:
def relative_error_direct(A, Ak):
    # YOUR CODE HERE
    # (⌐■_■)
    pass


def relative_error_from_singular_values(s, k):
    # YOUR CODE HERE
    # (❍ᴥ❍ʋ)
    pass


In [ ]:
# Проверка для какого-то из изображений
name = "C: фоточка"
img = images[name]
U, s, Vt = svds[name]

for k in [1, 5, 10, 20, 50]:
    Ak = reconstruct_from_svd(U, s, Vt, k)
    err_direct = relative_error_direct(img, Ak)
    err_spectrum = relative_error_from_singular_values(s, k)
    print(k, err_direct, err_spectrum)

__(0.75 балла) Задание 5:__ На одном графике нарисуйте зависимость относительной ошибки от $k$ для всех четырёх изображений.

In [ ]:
def plot_relative_errors(images, svds, max_k=120):
    # YOUR CODE HERE
    # [̲̅$̲̅(̲̅5̲̅)̲̅$̲̅]
    pass

plot_relative_errors(images, svds)

**Вопрос:** Проинтерпретируйте графики, какие картинки сжимаются лучше?

__Ответ:__ ...

__(1 балл) Задание 6:__ Представьте, что вы проектируете очень простой алгоритм сжатия изображений. Для каждого изображения выберите минимальное $k$, при котором относительная ошибка меньше 10% и заполните таблицу:

| изображение | минимальное k | коэффициент сжатия | комментарий |
|---|---:|---:|---|
| белый шум | | | |
| QR-like код | | | |
| фотография | | | |
| страница с текстом | | | |

Что интересного вы заметили, заполняя эту таблицу?

__Ответ:__

In [ ]:
# YOUR CODE HERE
#  ლ(ಠ益ಠლ)

# Да-да, для поиска корректных k и compression_ratio придется написать цикл

__(0.75 балла) Задание 7:__ Теперь сделаем главное упражнение первой части.

Постройте четыре спектра, но перемешайте подписи. Попробуйте по одному только спектру понять, какому изображению он принадлежит.

Идея:

* резкий спад спектра означает сильную структуру изображения;
* медленный спад означает, что информация распределена по многим компонентам;
* почти плоский спектр часто указывает на шум.

In [ ]:
def plot_shuffled_spectra(svds, seed=123):
    rng = np.random.default_rng(seed)
    items = list(svds.items())
    rng.shuffle(items)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    mapping = {}

    for idx, (name, (U, s, Vt)) in enumerate(items, start=1):
        label = f"Spectrum {idx}"
        mapping[label] = name

        axes[0].plot(s, label=label)
        axes[1].semilogy(s, label=label)

    axes[0].set_title("Обычная шкала")
    axes[1].set_title("Логарифмическая шкала")

    for ax in axes:
        ax.set_xlabel("i")
        ax.set_ylabel(r"$\sigma_i$")
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.show()

    return mapping

hidden_mapping = plot_shuffled_spectra(svds)

Заполните таблицу:

| Spectrum | изображение | почему |
|---|---|---|
| Spectrum 1 | | |
| Spectrum 2 | | |
| Spectrum 3 | | |
| Spectrum 4 | | |

После этого можно раскрыть правильный ответ следующей ячейкой. Не бойтесь не угадать, баллы за это не снимаются!

In [ ]:
# Раскомментируйте после того, как сделали предположение.
# hidden_mapping

__(0.5 балла) Задание 8:__ SVD тесно связано со спектральным разложением симметричных матриц. Если

$$
A = U \Sigma V^T,
$$

то

$$
A^T A = V \Sigma^2 V^T,
$$

а

$$
A A^T = U \Sigma^2 U^T.
$$

То есть:

* правые сингулярные векторы — собственные векторы матрицы $A^T A$;
* левые сингулярные векторы — собственные векторы матрицы $A A^T$;
* собственные числа этих матриц равны $\sigma_i^2$.


Возьмите одну картинку и проверьте численно, что собственные числа матрицы $A^T A$ совпадают с квадратами сингулярных чисел.

In [ ]:
small = transform.resize(images["C: фоточка"], (64, 64), anti_aliasing=True)

# 1. Посчитайте SVD маленькой картинки.
# 2. Посчитайте собственные числа матрицы small.T @ small.
# 3. Сравните их с квадратами сингулярных чисел.
# для симметричных матриц лучше использовать np.linalg.eigh

# YOUR CODE HERE
# ヾ(๑╹◡╹)ﾉ

---

# Часть 2: SVD превращается в PCA

В первой части мы смотрели на одно изображение как на матрицу пикселей:

$$
\text{строки} \times \text{столбцы}.
$$

Теперь поменяем точку зрения. Будем смотреть не на одну картинку, а на много маленьких изображений цифр.

В датасете `digits` каждая строка — это отдельная цифра размером $8 \times 8$, вытянутая в вектор длины 64:

$$
X \in \mathbb{R}^{n \times 64}.
$$

То есть теперь матрица устроена так:

$$
\text{объекты} \times \text{признаки}.
$$

SVD снова будет искать структуру, но теперь не внутри одной картинки, а внутри облака точек.

Именно так рождается метод главных компонент, PCA: мы ищем новые координаты, в которых данные меняются сильнее всего.

> В этой части не надо пользоваться `sklearn.decomposition.PCA`.  
> Задача — руками увидеть, как PCA получается из SVD.

Мы сделаем четыре шага:

1. загрузим картинки цифр и поймём, как устроена матрица данных;
2. применим SVD к нецентрированной матрице и увидим, почему это не совсем PCA;
3. вычтем среднюю цифру и построим настоящую PCA-проекцию;
4. посмотрим на главные компоненты как на картинки и поймём, какие признаки они ловят.

__(0 баллов) Задание 9:__ Загрузите датасет `digits` из `sklearn.datasets`.

Ответьте на вопросы:

1. Сколько объектов в датасете?
2. Сколько признаков у каждого объекта?
3. Почему признаков именно столько?
4. Что означает одна строка матрицы `X`?

In [ ]:
from sklearn.datasets import load_digits

# YOUR CODE HERE
# (╯°□°)╯︵ ┻━┻

digits = ...
X = ...
y = ...

print(X.shape)
print(y.shape)

Сначала сделаем почти то же самое, что в первой части: применим SVD прямо к матрице `X`.

$$
X = U \Sigma V^T.
$$

Если взять первые две колонки матрицы $U \Sigma$, мы получим двумерные координаты объектов.

Это похоже на PCA, но есть важная проблема: мы пока не вычли средний объект.  
Поэтому первое направление может описывать не форму различий между цифрами, а просто «среднюю яркость» или общий сдвиг данных.


__(0.75 балла) Задание 10:__

1. посчитайте SVD матрицы `X` без центрирования;
2. постройте график сингулярных чисел;
3. спроецируйте данные на первые две SVD-координаты;
4. покрасьте точки по настоящим цифрам `y`.

In [ ]:
def svd_coordinates(X, n_components=2, center=False):
    # Возвращает координаты объектов в первых n_components SVD-направлениях.
    #
    # Если center=True, сначала нужно вычесть среднее по каждому признаку.
    # Координаты можно получить как U[:, :k] * s[:k].

    # YOUR CODE HERE
    # (⌐■_■)
    pass


def plot_embedding(Z, y, title):
    # Рисует двумерную проекцию данных.
    # Можно начать с обычного scatter plot.

    # YOUR CODE HERE
    # (☞ﾟヮﾟ)☞
    pass


# YOUR CODE HERE
# ʕ•ᴥ•ʔ

Z_raw = ...
plot_embedding(Z_raw, y, title="SVD без центрирования")

PCA ищет направления максимальной изменчивости **относительно среднего объекта**.

Поэтому перед SVD надо вычесть среднее значение каждого признака:

$$
X_c = X - \bar X.
$$

Здесь $\bar X$ — средняя цифра. Это вектор длины 64, где каждый элемент показывает среднюю яркость соответствующего пикселя.

__(0.75 балла) Задание 11:__

1. посчитайте среднюю цифру и нарисуйте её как картинку $8 \times 8$;
2. вычтите среднюю цифру из всех объектов;
3. примените SVD к центрированной матрице;
4. постройте двумерную PCA-проекцию.

In [ ]:
# 1. Посчитайте и нарисуйте среднюю цифру

# YOUR CODE HERE
# (づ｡◕‿‿◕｡)づ

In [ ]:
# 2. Получите PCA-проекцию после центрирования и нарисуйте её

# YOUR CODE HERE
# (ﾉ◕ヮ◕)ﾉ*:･ﾟ✧

Z_centered = ...
plot_embedding(...)

__(0.75 балла) Задание 12:__ Сравните две визуализации:

1. SVD без центрирования;
2. SVD после центрирования.

Ответьте на вопросы:

Где цифры разделяются лучше?

__Ответ:__

Почему без центрирования первая компонента может быть не очень полезной?

__Ответ:__

Что означает «средняя цифра», которую мы вычли?

__Ответ:__

__(0.75 балла) Задание 13:__

В первой части SVD раскладывало одну картинку на простые картинки ранга 1.

В PCA всё похоже, но теперь строки матрицы $V^T$ можно интерпретировать как **новые признаки**.  
Каждая главная компонента — это направление в 64-мерном пространстве пикселей. Значит, её тоже можно нарисовать как картинку $8 \times 8$.

Нарисуйте первые 10 главных компонент для центрированной матрицы.

In [ ]:
def show_components(Vt, n_components=10, image_shape=(8, 8)):
    # Нарисуйте первые n_components строк Vt как изображения.
    # component = Vt[i].reshape(image_shape)
    # Для визуализации полезно использовать cmap="coolwarm",
    # потому что значения могут быть положительными и отрицательными.

    # YOUR CODE HERE
    # ∑(°ロ°)
    pass


# YOUR CODE HERE
# (☉_☉)

show_components(...)

Ответьте на вопросы:

Похожи ли главные компоненты на обычные цифры?

__Ответ:__

Какие зоны изображения они выделяют?

__Ответ:__

Почему компоненты могут содержать как положительные, так и отрицательные значения?

__Ответ:__

Чем эти картинки отличаются от реконструкций из первой части?

__Ответ:__


В первой части мы выбирали число компонент по ошибке восстановления изображения.

В PCA обычно смотрят на **долю объяснённой дисперсии**.

Если сингулярные числа центрированной матрицы равны $\sigma_1, \ldots, \sigma_r$, то доля дисперсии, объяснённая $i$-й компонентой, равна

$$
\frac{\sigma_i^2}{\sum_j \sigma_j^2}.
$$

А накопленная доля для первых $k$ компонент:

$$
\frac{\sigma_1^2 + \ldots + \sigma_k^2}{\sum_j \sigma_j^2}.
$$


__(1 балл) Задание 14:__

1. постройте график с накопленной долей объяснённой дисперсии;
2. найдите минимальное число компонент для 80%, 90% и 95% объяснённой дисперсии;
3. восстановите несколько цифр по $k = 2, 8, 16, 32$ PCA-компонентам.

In [ ]:
# YOUR CODE HERE
# (｡•̀ᴗ-)✧

__[1 балл (бонус)]__  Сравните работу SVD с другим методом понижения размерности: [случайными гауссовскими проекциями](http://scikit-learn.org/stable/modules/generated/sklearn.random_projection.GaussianRandomProjection.html).

In [ ]:
# YOUR CODE HERE
# (❍ᴥ❍ʋ)

**Подробное описание вашего сравнения и сделанные выводы:**

.....